# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [4]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [21]:
context = [ {'role':'system', 'content':"""
 You are a Text-to-SQL assistant. For each user question, return **one SQL query** using ONLY the schema and sample data below.

## Rules
1) Write only the SQL (no explanations, no markdown).
2) Use only the tables and columns given here.
3) Years are stored as integers (e.g., 2025). Use numeric comparisons like year = 2025.
4) If the question is not clear, choose the simplest way.
5) When using aggregates, include all non-aggregated columns in GROUP BY.
6) When using more than one table, use JOIN ... ON ... to connect them.
7) 7) If the question cannot be answered with this schema, return:
   SELECT 'Not answerable with given schema' AS message;

## Tables and Sample Data

-- employees
CREATE TABLE employees (
  ID_usr INTEGER PRIMARY KEY,
  name   VARCHAR(100)
);

INSERT INTO employees VALUES
  (1, 'Alice Smith'),
  (2, 'Bob Johnson'),
  (3, 'Carla Rossi'),
  (4, 'Daniel Weber'),
  (5, 'Eva Novak');

-- salary
CREATE TABLE salary (
  ID_usr INTEGER,
  year   INTEGER,
  salary DECIMAL(12,2),
  PRIMARY KEY (ID_usr, year)
);

INSERT INTO salary VALUES
  (1, 2023, 52000.00),
  (1, 2024, 56000.00),
  (1, 2025, 59000.00),
  (2, 2023, 48000.00),
  (2, 2024, 50000.00),
  (3, 2024, 72000.00),
  (3, 2025, 75000.00),
  (4, 2023, 61000.00),
  (4, 2024, 61000.00),
  (5, 2025, 45000.00);

-- studies
CREATE TABLE studies (
  ID         INTEGER PRIMARY KEY,
  ID_usr     INTEGER,
  educational_level VARCHAR(40),
  institution       VARCHAR(120),
  start_year  INTEGER,
  end_year    INTEGER,
  speciality  VARCHAR(120)
);

INSERT INTO studies VALUES
  (101, 1, 'Bachelor', 'TU Berlin',          2016, 2019, 'Computer Science'),
  (102, 1, 'Master',   'TU Berlin',          2019, 2021, 'Data Science'),
  (103, 2, 'Bachelor', 'LMU Munich',         2015, 2018, 'Economics'),
  (104, 3, 'Master',   'Sapienza Rome',      2017, 2019, 'Statistics'),
  (105, 3, 'PhD',      'Sapienza Rome',      2019, 2023, 'Machine Learning'),
  (106, 4, 'Bachelor', 'Uni Hamburg',        2012, 2015, 'Information Systems'),
  (107, 5, 'Bachelor', 'Charles University', 2018, 2021, 'Business Analytics');
"""}]

In [6]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [7]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [24]:
#new
context_user = context.copy()
print(return_CCRMSQL("Return all employees.", context_user))

SELECT * FROM employees;


In [25]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Return all employees.", old_context_user))

This is your SQL:
```sql
SELECT * FROM employees;
```
This SQL command retrieves all the employees from the "employees" table.


In [26]:
#new
print(return_CCRMSQL("Return all salaries in 2024.", context_user))

SELECT * 
FROM salary
WHERE year = 2024;


In [27]:
#old
print(return_CCRMSQL("Return all salaries in 2024.", old_context_user))

This is your SQL:
```sql
SELECT * FROM salary WHERE year = '2024';
```
This SQL command selects all entries from the "salary" table where the year is 2024, returning all salaries for that year.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [28]:
#new
print(return_CCRMSQL("Return list employees and their studies.", context_user))

SELECT e.name, s.educational_level, s.institution, s.start_year, s.end_year, s.speciality
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr;


In [29]:
#old
print(return_CCRMSQL("Return list employees and their studies.", old_context_user))

This is your SQL:
```sql
SELECT employees.name, studies.educational_level, studies.Institution, studies.Years, studies.Speciality
FROM employees
JOIN studies ON employees.ID_usr = studies.ID_usr;
```

This SQL query retrieves the names of employees along with their educational level, institution, years of study, and specialty by joining the "employees" table with the "studies" table on the common ID_usr field.


In [30]:
#new
print(return_CCRMSQL("Return who earned the highest salary in 2025?", context_user))

SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = 2025
ORDER BY s.salary DESC
LIMIT 1;


In [31]:
#old
print(return_CCRMSQL("Return who earned the highest salary in 2025?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2025'
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query retrieves the name of the employee who earned the highest salary in the year 2025 by joining the "employees" and "salary" tables on the employee ID. It then filters the results to only include salaries from 2025, orders them in descending order, and limits the output to the top result.


In [32]:
#new
print(return_CCRMSQL("Return each employee with their latest salary.", context_user))

SELECT e.ID_usr, e.name, s.salary
FROM employees e
JOIN (
    SELECT ID_usr, MAX(year) AS latest_year
    FROM salary
    GROUP BY ID_usr
) latest_salary
ON e.ID_usr = latest_salary.ID_usr
JOIN salary s
ON e.ID_usr = s.ID_usr AND s.year = latest_salary.latest_year;


In [33]:
#old
print(return_CCRMSQL("Return each employee with their latest salary.", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = (SELECT MAX(year) FROM salary WHERE ID_usr = e.ID_usr);
```

This SQL query retrieves each employee's name along with their latest salary by joining the "employees" and "salary" tables on the employee ID. It then filters the results to only include the rows where the salary year matches the maximum year for that employee, effectively showing their latest salary.


In [34]:
#new
print(return_CCRMSQL("Return list employees with more than one degree.", context_user))

SELECT e.ID_usr, e.name
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
GROUP BY e.ID_usr, e.name
HAVING COUNT(s.ID) > 1;


In [35]:
#old
print(return_CCRMSQL("Return list employees with more than one degree.", old_context_user))

This is your SQL:
```sql
SELECT e.ID_usr, e.name
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
GROUP BY e.ID_usr, e.name
HAVING COUNT(DISTINCT s.ID) > 1;
```

This SQL query retrieves a list of employees who have more than one degree by joining the employees table with the studies table on the employee ID. It then groups the results by employee ID and name, and filters the results to only include employees with more than one distinct degree using the HAVING clause with the COUNT function.
